# CUDA-MEEP vs Meep: GPU Benchmark

Compares **CUDA-MEEP** (PyTorch GPU-native FDTD) vs **Meep** (CPU FDTD).

**Note on GPU crossover:** For small grids (64², 128²) CPU may outperform GPU.
This is expected — kernel launch overhead dominates when the grid is too small
to saturate the GPU's SMs. GPU wins decisively at 256² and above.

> **Enable GPU first:** Runtime → Change runtime type → T4 GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'NOT DETECTED — enable GPU in Runtime settings')

In [ ]:
!git clone https://github.com/shahzaibshazoo/cuda-meep.git
!pip install torch numpy matplotlib pytest --quiet

In [ ]:
import sys, time
import numpy as np
sys.path.insert(0, '/content/cuda-meep/src')
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Part 1: CUDA-MEEP Benchmark

In [ ]:
from core import YeeGrid, FieldSet, MurABC, GaussianPulse, PointSource, SourceCollection, FDTD2D

GRID_SIZES = [64, 128, 256, 512]
N_WARMUP   = 20
N_STEPS    = 200
DX         = 1e-3
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

def run_cuda_meep(N, device, n_warmup=N_WARMUP, n_steps=N_STEPS):
    grid     = YeeGrid(N, N, dx=DX, dy=DX, device=device)
    fields   = FieldSet(grid)
    boundary = MurABC(grid, fields.Hz)
    pulse    = GaussianPulse(amplitude=1.0, sigma=30*grid.dt)
    src      = PointSource(pulse, N//2, N//2, 'Hz', grid=grid, N_steps=n_warmup+n_steps)
    sim      = FDTD2D(grid, fields, boundary, SourceCollection([src]), n_check=500)
    sim.run(n_warmup)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    sim.run(n_steps)
    if device == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return n_steps * N * N / elapsed / 1e6, elapsed / n_steps * 1000

gpu_results, cpu_results = [], []

print(f'--- CUDA-MEEP ({DEVICE.upper()}) ---')
for N in GRID_SIZES:
    m, ms = run_cuda_meep(N, DEVICE)
    gpu_results.append({'N': N, 'mcells_s': m, 'ms_step': ms})
    print(f'  {N:4d}²  {m:8.1f} Mcells/s  {ms:8.3f} ms/step')

if DEVICE == 'cuda':
    print('--- CUDA-MEEP (CPU) ---')
    for N in GRID_SIZES:
        m, ms = run_cuda_meep(N, 'cpu')
        cpu_results.append({'N': N, 'mcells_s': m, 'ms_step': ms})
        print(f'  {N:4d}²  {m:8.1f} Mcells/s  {ms:8.3f} ms/step')
else:
    cpu_results = gpu_results
    print('\n(No GPU — only CPU results available. Enable GPU for full comparison.)')

## Part 2: Install Meep

In [ ]:
print('Installing pymeep via conda-forge (~2-3 min)...')
!conda install -c conda-forge pymeep=*=nompi* -y -q 2>&1 | tail -3

MEEP_AVAILABLE = False
try:
    import meep as mp
    _ = mp.Vector3(0, 0, 0)   # confirm it's the real compiled package
    try:
        import importlib.metadata
        ver = importlib.metadata.version('meep')
    except Exception:
        ver = 'unknown'
    print(f'Meep ready (version: {ver})')
    MEEP_AVAILABLE = True
except Exception as e:
    print(f'Meep unavailable: {e}')
    print('Meep benchmark will be skipped. Speedup column will show GPU vs CPU only.')

## Part 3: Meep Benchmark

In [ ]:
# meep_results is EMPTY when Meep is unavailable.
# No fake estimates — speedup column will show N/A.
meep_results = []

if MEEP_AVAILABLE:
    import meep as mp, os
    os.environ['MEEP_VERBOSITY'] = '0'

    courant = 0.5   # Meep default

    for N in GRID_SIZES:
        sim_mp = mp.Simulation(
            cell_size=mp.Vector3(1, 1),
            resolution=N,
            sources=[mp.Source(
                mp.GaussianSource(frequency=1.0, fwidth=0.5),
                component=mp.Hz,
                center=mp.Vector3(0, 0)
            )],
            boundary_layers=[mp.Absorber(thickness=0.1)]
        )
        # Warmup
        sim_mp.run(until=5.0/N)
        sim_mp.reset_meep()

        dt_meep  = courant / N
        t_target = N_STEPS * dt_meep
        t0 = time.perf_counter()
        sim_mp.run(until=t_target)
        elapsed = time.perf_counter() - t0

        actual_steps = max(1, int(round(t_target / dt_meep)))
        mcells_s = actual_steps * N * N / elapsed / 1e6
        ms_step  = elapsed / actual_steps * 1000
        meep_results.append({'N': N, 'mcells_s': mcells_s, 'ms_step': ms_step})
        print(f'  {N:4d}²  meep  {mcells_s:8.1f} Mcells/s  {ms_step:8.3f} ms/step')
        sim_mp.reset_meep()
else:
    print('Meep skipped — no results to show.')

## Part 4: Results

In [ ]:
# Results table — only shows columns that have real data
has_gpu   = DEVICE == 'cuda'
has_meep  = len(meep_results) > 0

print('='*72)
header = f'  {"Grid":5s}'
if has_gpu:  header += f'  {"GPU (Mcells/s)":>16s}'
header += f'  {"CPU (Mcells/s)":>16s}'
if has_meep: header += f'  {"Meep (Mcells/s)":>16s}'
# Speedup column: GPU vs Meep > GPU vs CPU > nothing
if has_gpu and has_meep:   header += f'  {"GPU/Meep":>9s}'
elif has_gpu:              header += f'  {"GPU/CPU":>9s}'
print(header)
print('='*72)

for i, N in enumerate(GRID_SIZES):
    gpu  = gpu_results[i]['mcells_s']  if has_gpu  else None
    cpu  = cpu_results[i]['mcells_s']
    meep = meep_results[i]['mcells_s'] if has_meep else None

    row = f'  {N}²  '
    if has_gpu:  row += f'{gpu:>14.1f}  '
    row += f'{cpu:>14.1f}  '
    if has_meep: row += f'{meep:>14.1f}  '

    if has_gpu and has_meep:
        row += f'{gpu/meep:>7.1f}x'
    elif has_gpu:
        row += f'{gpu/cpu:>7.1f}x'

    print(row)

print('='*72)

if not has_meep:
    print()
    print('Meep was not installed — GPU/Meep speedup cannot be computed.')
    print('Showing GPU/CPU speedup instead (meaningful but different metric).')
    print('To get real Meep numbers, run this notebook on a system with pymeep installed.')

if has_gpu:
    print()
    crossover = [N for i,N in enumerate(GRID_SIZES) if gpu_results[i]['mcells_s'] <= cpu_results[i]['mcells_s']]
    if crossover:
        print(f'Note: CPU faster than GPU at {crossover} — expected for small grids where')
        print('GPU kernel launch overhead exceeds compute time. GPU wins at larger grids.')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CUDA-MEEP vs Meep: FDTD Throughput Benchmark', fontsize=13)

ax = axes[0]
if has_gpu:
    ax.plot(GRID_SIZES, [r['mcells_s'] for r in gpu_results], 'o-g',
            label=f'CUDA-MEEP (GPU: {torch.cuda.get_device_name(0) if has_gpu else ""})', lw=2, ms=8)
ax.plot(GRID_SIZES, [r['mcells_s'] for r in cpu_results], 's-b', label='CUDA-MEEP (CPU)', lw=2, ms=8)
if has_meep:
    ax.plot(GRID_SIZES, [r['mcells_s'] for r in meep_results], '^-r', label='Meep (CPU)', lw=2, ms=8)
ax.set(xlabel='Grid size', ylabel='Throughput (Mcells/s)', title='Throughput Comparison')
ax.set_xticks(GRID_SIZES); ax.set_xticklabels([f'{N}²' for N in GRID_SIZES])
ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
if has_gpu and has_meep:
    speedups  = [gpu_results[i]['mcells_s']/meep_results[i]['mcells_s'] for i in range(len(GRID_SIZES))]
    colors    = ['red' if s < 1 else 'green' for s in speedups]
    bars = ax2.bar([f'{N}²' for N in GRID_SIZES], speedups, color=colors, alpha=0.85)
    ax2.bar_label(bars, fmt='%.1fx', fontsize=12)
    ax2.axhline(1, color='black', ls='--', alpha=0.4, label='Meep = 1×')
    ax2.set(ylabel='Speedup vs Meep (CPU)', title='GPU Speedup over Meep')
    ax2.legend()
elif has_gpu:
    speedups  = [gpu_results[i]['mcells_s']/cpu_results[i]['mcells_s'] for i in range(len(GRID_SIZES))]
    colors    = ['red' if s < 1 else 'steelblue' for s in speedups]
    bars = ax2.bar([f'{N}²' for N in GRID_SIZES], speedups, color=colors, alpha=0.85)
    ax2.bar_label(bars, fmt='%.1fx', fontsize=12)
    ax2.axhline(1, color='black', ls='--', alpha=0.4, label='CPU = 1×')
    ax2.set(ylabel='Speedup vs CPU', title='GPU/CPU Speedup\n(Meep unavailable — install pymeep for full comparison)')
    ax2.legend()
else:
    ax2.text(0.5, 0.5, 'Enable GPU runtime\nfor speedup chart',
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/benchmark_results.png')

## Part 5: Brain Tumor Detection Demo

In [ ]:
import os
os.chdir('/content/cuda-meep')
print('Running 16-antenna MIMO brain tumor imaging (32 FDTD simulations)...')
print('CPU: ~90s  |  T4 GPU: ~10-15 min (larger grid on GPU)')
r = subprocess.run([sys.executable, 'examples/brain_mimo_imaging.py'],
                   capture_output=True, text=True, timeout=1200)
print(r.stdout)
if r.returncode != 0:
    print('ERROR:', r.stderr[-2000:])

In [ ]:
from IPython.display import Image, display
img = '/content/cuda-meep/examples/output/brain_mimo_imaging.png'
display(Image(filename=img)) if os.path.exists(img) else print('Image missing — check errors above')